# ============================================================
# Kaggle Notebook: DKVMN-style (topic-level adaptation), 2k, 5 folds x 50 epochs
# ============================================================
# RUN IN ORDER. 2k config ONLY (max-seq-len 2000), never 10k.
# Expected runtime: ~4-8 hours on T4 x1 (see epoch-cap rule below)
# Accelerator recommendation: GPU T4 x1
#
# One-time setup per notebook: Add Data -> training.csv dataset,
# `dkt-weights-5fold` dataset, AND `cpcoach-alt-code` dataset
# (built locally by: python3 training/_gen_alt_notebooks.py --pack).
# ============================================================

### CELL 0 — Install dependencies


In [ ]:
# Install torch-geometric so the Graph-DKT path is activated.
# The dense GCN uses pure PyTorch (einsum) under the hood, so heavy binary dependencies
# (like pyg_lib, torch_scatter, torch_sparse, torch_cluster) are NOT required.
# Kaggle already has a fully-functional PyTorch + CUDA pre-installed.
# Note: You MUST toggle 'Internet on' in the right panel under Settings to download packages!
try:
    import torch_geometric
    print("torch_geometric is already installed!")
except ImportError:
    print("Installing torch-geometric...")
    !pip install torch-geometric -q || echo "WARNING: Could not install torch-geometric. Check if Internet is toggled ON in the right panel under Settings. Falling back to pure PyTorch dense GCN mode."

!pip install scikit-learn pandas tqdm -q

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    try:
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    except AttributeError:
        props = torch.cuda.get_device_properties(0)
        mem = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
        print(f"VRAM: {mem / 1e9:.1f} GB")

### CELL 1 — Copy training.csv from Kaggle input


In [ ]:
# When you use Kaggle's "Add Data" → "Upload", the file
# lands in /kaggle/input/<dataset-name>/training.csv

import os, shutil

# Find training.csv in Kaggle input directory
input_dir = "/kaggle/input"
training_csv = None
for root, dirs, files in os.walk(input_dir):
    if "training.csv" in files:
        training_csv = os.path.join(root, "training.csv")
        break

if training_csv is None:
    print("ERROR: training.csv not found in Kaggle input!")
    print("Go to: Add data → Upload → select training.csv")
    print("Then re-run this cell.")
    import sys; sys.exit(1)

print(f"Found: {training_csv}")
print(f"Size: {os.path.getsize(training_csv) / 1e6:.1f} MB")

### CELL 2 — Clone repo and copy training data


In [ ]:
!git clone https://github.com/MannPatel1236/CP-Coach.git /kaggle/working/CP-Coach

# Copy training.csv into repo
data_dir = "/kaggle/working/CP-Coach/backend/data"
os.makedirs(data_dir, exist_ok=True)
shutil.copy(training_csv, os.path.join(data_dir, "training.csv"))

print(f"Copied training.csv to {data_dir}")
!wc -l /kaggle/working/CP-Coach/backend/data/training.csv

In [ ]:
# NEW CODE OVERLAY (local changes are NOT pushed to GitHub).
# Dataset: private `cpcoach-alt-code` (= backend/kaggle_alt_code.zip).
# Kaggle keeps an uploaded .zip UNEXTRACTED, so this cell unzips first.
import glob, os, shutil, sys, zipfile

print('--- /kaggle/input contents ---')
for _d in sorted(glob.glob('/kaggle/input/*')):
    try:
        print(' ', _d, '->', sorted(os.listdir(_d))[:10])
    except OSError:
        print(' ', _d, '-> (unlistable)')

def _norm(_s): return _s.lower().replace('-', '').replace('_', '')

target = None
for _d in glob.glob('/kaggle/input/*'):
    _n = _norm(os.path.basename(_d))
    if 'cpcoach' in _n and 'alt' in _n and 'code' in _n:
        target = _d
        break
if target is None:
    raise FileNotFoundError('Overlay dataset not found: expected a dir like cpcoach-alt-code under /kaggle/input/ (see listing above).')
print(f'overlay dataset dir: {target}')

srcs = [target]
_tmp = '/tmp/altcode_unzipped'
for _z in glob.glob(os.path.join(target, '*.zip')):
    with zipfile.ZipFile(_z) as _zf:
        _zf.extractall(_tmp)
    print(f'extracted {_z} -> {_tmp}')
    srcs.append(_tmp)

found = False
for _src in srcs:
    for _root, _, _files in os.walk(_src):
        for _fn in _files:
            if _fn.endswith('.py'):
                _s = os.path.join(_root, _fn)
                _rel = os.path.relpath(_s, _src)
                _dst = os.path.join('/kaggle/working/CP-Coach/backend', _rel)
                os.makedirs(os.path.dirname(_dst), exist_ok=True)
                shutil.copy(_s, _dst)
                print(f'Overlaid: {_rel}')
                found = True
if not found:
    raise FileNotFoundError('No .py files inside the overlay dataset (checked dir + any zips).')
sys.path.insert(0, '/kaggle/working/CP-Coach/backend')

# Verify the new code is importable with the new flags.
from models.sakt import SAKTStyleModel
from models.dkvmn import DKVMNStyleModel
import training.train_dkt as T
assert '--model' in open('training/train_dkt.py').read() and "'sakt'" in open('training/train_dkt.py').read()
print('overlay OK: sakt/dkvmn importable, train_dkt has new models')


### CELL 3 — Verify data + quick stats


In [ ]:
%cd /kaggle/working/CP-Coach/backend

import sys
sys.path.insert(0, "/kaggle/working/CP-Coach/backend")

from data.topic_graph import CPTopicGraph
from training.train_dkt import load_csv_sequences

tg = CPTopicGraph()
# Let's load with max_seq_len=2000 matching our training configurations
seqs = load_csv_sequences("data/training.csv", tg, max_seq_len=2000)
print(f"Loaded {len(seqs)} user sequences (max_seq_len=2000)")

import numpy as np
lengths = np.array([len(s) for s in seqs])
print(f"Sequence length stats (after truncation):")
for p in [50, 75, 90, 95, 99, 100]:
    print(f"  p{p}: {np.percentile(lengths, p):.0f}")

### Smoke test for dkvmn (3 epochs, 1 fold)
**EPOCH-CAP RULE (DKVMN only):** the per-step Python loop makes DKVMN slower than DKT. After the smoke cell, read s/epoch, project 5 folds x 50 epochs; if the projection exceeds ~8 GPU-h, lower --epochs in the train cell (e.g. 25) AND record the cap — the paper states the cap identically for this model only.

In [ ]:
%cd /kaggle/working/CP-Coach/backend
!python3 -m training.train_dkt \
  --data data/training.csv --model dkvmn \
  --epochs 3 --folds 1 --out weights/smoke_dkvmn.pt \
  --batch 16 --device cuda --max-seq-len 2000 --seed 42

# Smoke must show: loss decreasing, s/epoch, val AUC. For dkvmn, project 5 folds x 50 epochs from s/epoch here.


### CELL 5 — Restore pre-trained DKT weights (no retrain)

**Why:** DKT retraining takes ~1h per notebook. We've already trained 5-fold DKT
weights elsewhere (the canonical baseline at AUC 0.9674). Upload those once as a
private Kaggle dataset (`dkt-weights-5fold`) and this notebook restores them in seconds.

Saves ~1h × 3 ablation notebooks = ~3h of GPU time across the whole ablation study.

In [ ]:
# Restore pre-trained 5-fold DKT weights from the Kaggle input dataset.
# Dataset name convention: 'dkt-weights-5fold' (upload weights/dkt_fold{0-4}.pt there once).
import glob, os, shutil

os.makedirs('weights', exist_ok=True)
found_any = False
for d in glob.glob('/kaggle/input/dkt-weights-5fold*'):
    for f in glob.glob(os.path.join(d, 'dkt_fold*.pt')):
        shutil.copy(f, 'weights/')
        print(f'Restored: {f}')
        found_any = True

if not found_any:
    raise FileNotFoundError(
        "DKT weights dataset not found in /kaggle/input/.\n"
        "Upload weights/dkt_fold{0,1,2,3,4}.pt as a private Kaggle dataset named\n"
        "'dkt-weights-5fold', then 'Add Data → Your Datasets' before running this cell."
    )

restored = sorted(glob.glob('weights/dkt_fold*.pt'))
print(f'\nRestored {len(restored)} DKT fold weights: {[os.path.basename(f) for f in restored]}')


### Full train: dkvmn (5 folds x 50 epochs)
**EPOCH-CAP RULE (DKVMN only):** the per-step Python loop makes DKVMN slower than DKT. After the smoke cell, read s/epoch, project 5 folds x 50 epochs; if the projection exceeds ~8 GPU-h, lower --epochs in the train cell (e.g. 25) AND record the cap — the paper states the cap identically for this model only.

In [ ]:
# Full train. Run in foreground so Save & Run All blocks.
# Expected runtime: ~4-8 hours on T4 x1 (see epoch-cap rule below).

!python3 -m training.train_dkt \
  --data data/training.csv --model dkvmn \
  --epochs 50 --folds 5 --out weights/dkvmn.pt \
  --batch 16 --device cuda --max-seq-len 2000 --seed 42


### CELL 7 — Compare results


In [ ]:
# Stage dkvmn_foldN.pt -> graph_dkt_foldN.pt for eval_folds.py.
!for i in 0 1 2 3 4; do cp weights/dkvmn_fold${i}.pt weights/graph_dkt_fold${i}.pt; done

# Evaluate (GPU box has no spare CPU issue; device cuda is fine).
!python3 -m training.eval_folds --device cuda --max-seq-len 2000 --folds 5 --model-type dkvmn --sidecar results/eval_dkvmn_2k_perfolds.sidecar.json --per-user-npz results/eval_dkvmn_2k_perfolds.peruser.npz | tee results/eval_dkvmn_2k_perfolds.txt


### CELL 8 — Download results to your machine


In [ ]:
import zipfile, glob

results = []
for pat in ['weights/dkvmn_fold*.pt',
            'results/eval_dkvmn_2k_perfolds.*']:
    for f in glob.glob(pat):
        results.append(f)
        print(f'Packed: {f}')
with zipfile.ZipFile('/kaggle/working/results.zip', 'w') as z:
    for f in results:
        z.write(f)
print('\nDone! Download /kaggle/working/results.zip from the Output panel.')
